# Aula 15 - Notebook: Simulação de Injeção de Falhas e Desvio Automático de Fluxo

Neste notebook simulamos uma contingência em tempo real: o travamento mecânico (jamming) de uma esteira principal e o recálculo dinâmico de rota pelo SCADA para direcionar as peças por uma esteira de desvio (bypass) até a caixa de loteamento.


In [1]:
def formatar_tabela(dados):
    if not dados: return 'Tabela Vazia'
    cols = list(dados[0].keys())
    w = {c: max(len(c), max(len(str(r.get(c, ''))) for r in dados)) for c in cols}
    head = ' | '.join(f'{c:<{w[c]}}' for c in cols)
    div = '-+-'.join('-' * w[c] for c in cols)
    rows = [' | '.join(f'{str(r.get(c, '')):<{w[c]}}' for c in cols) for r in dados]
    return '\n'.join([head, div] + rows)

class GrafoEsteiras:
    def __init__(self, vertices):
        self.vertices = vertices
        self.v2i = {v: i for i, v in enumerate(vertices)}
        self.i2v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n): self.pesos[i][i] = 0.0

    def add_esteira(self, u, v, comp):
        self.pesos[self.v2i[u]][self.v2i[v]] = comp

def criar_rede_manufatura():
    g = GrafoEsteiras(['Silo', 'Inspecao', 'Divisao_Principal', 'Esteira_A', 'Esteira_B', 'Caixa_Lote'])
    g.add_esteira('Silo', 'Inspecao', 5.0)
    g.add_esteira('Inspecao', 'Divisao_Principal', 2.0)
    g.add_esteira('Divisao_Principal', 'Esteira_A', 3.0)  # Rota Primária
    g.add_esteira('Divisao_Principal', 'Esteira_B', 8.0)  # Rota Secundária/Bypass
    g.add_esteira('Esteira_A', 'Caixa_Lote', 10.0)
    g.add_esteira('Esteira_B', 'Caixa_Lote', 12.0)
    return g

import time, heapq
class SCADARoteador:
    def __init__(self, g): self.g = g
    def dijkstra(self, src, dst):
        dist = {v: float('inf') for v in self.g.vertices}
        pred = {v: None for v in self.g.vertices}
        dist[src] = 0.0
        pq = [(0.0, src)]
        while pq:
            d, u = heapq.heappop(pq)
            if d > dist[u]: continue
            if u == dst: break
            ui = self.g.v2i[u]
            for vi in range(self.g.n):
                peso = self.g.pesos[ui][vi]
                if peso < float('inf'):
                    v = self.g.i2v[vi]
                    if dist[u] + peso < dist[v]:
                        dist[v] = dist[u] + peso
                        pred[v] = u
                        heapq.heappush(pq, (dist[v], v))
        path, curr = [], dst
        while curr: path.append(curr); curr = pred[curr]
        return dist[dst], path[::-1]

    def injetar_falha_recalcular(self, u, v, src, dst):
        t0 = time.perf_counter()
        # Injeção da falha (peso infinito)
        self.g.pesos[self.g.v2i[u]][self.g.v2i[v]] = float('inf')
        # Recálculo
        custo, caminho = self.dijkstra(src, dst)
        t_ms = (time.perf_counter() - t0) * 1000.0

        return {
            'Trecho_Travado': f'{u} -> {v}',
            'Nova_Rota_Ativa': ' -> '.join(caminho),
            'Comprimento_Total': f'{custo:.1f} m',
            'Tempo_Decisão_ms': f'{t_ms:.4f}'
        }

scada = SCADARoteador(criar_rede_manufatura())
# Simula travamento na Esteira_A (rota principal)
resultado = scada.injetar_falha_recalcular('Esteira_A', 'Caixa_Lote', 'Silo', 'Caixa_Lote')

print('=== RELATÓRIO SCADA: DESVIO DE PRODUÇÃO ===')
print(formatar_tabela([resultado]))


=== RELATÓRIO SCADA: DESVIO DE PRODUÇÃO ===
Trecho_Travado          | Nova_Rota_Ativa                                                  | Comprimento_Total | Tempo_Decisão_ms
------------------------+------------------------------------------------------------------+-------------------+-----------------
Esteira_A -> Caixa_Lote | Silo -> Inspecao -> Divisao_Principal -> Esteira_B -> Caixa_Lote | 27.0 m            | 0.0462          
